# Keras → OpenVINO 변환기

`weights/leather_model.keras` (VGG16 기반 가죽 결함 이진분류, 224×224, sigmoid 단일 출력)를
OpenVINO IR(`.xml` + `.bin`)로 변환한다.

- **셀 A**: 변환 → `model/leather_model.xml`, `model/leather_model.bin` 생성 (필수)
- **셀 B**: Keras 원본과 OpenVINO 결과 비교 (검증용, 선택)

실행 위치는 `inspection_app_ov/` 폴더. 실제 추론 코드는 `openvino_infer_ov.py` / `ov_ui.py` 로 들어간다.

In [ ]:
# # 변환에만 필요 (배포 requirements.txt 에는 tensorflow 를 넣지 않는다)
# %pip install -q "openvino==2026.3.1" "tensorflow==2.17.0" "numpy==1.26.4" "pillow"

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [3]:
import os
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
# ⚠ TF_USE_LEGACY_KERAS 는 설정하지 않는다 (leather_model.keras 는 Keras 3 포맷)

import numpy as np
from PIL import Image
import openvino as ov
import tensorflow as tf

print("OpenVINO   :", ov.__version__)
print("TensorFlow :", tf.__version__)

OpenVINO   : 2026.3.1-22476-759c5a6ab8c-releases/2026/3
TensorFlow : 2.17.0


## 셀 A — 변환 (xml · bin 생성)

In [4]:
KERAS_PATH      = Path("weights/leather_model.keras")
SAVED_MODEL_DIR = Path("model/leather_saved_model")   # 변환용 중간 산출물 (배포 제외)
IR_PATH         = Path("model/leather_model.xml")
IR_PATH.parent.mkdir(exist_ok=True)

# 1) .keras → SavedModel  (Keras 3 은 save() 가 아니라 export())
model = tf.keras.models.load_model(KERAS_PATH)
model.export(SAVED_MODEL_DIR)

# 2) SavedModel → OpenVINO IR  (xml + bin, 기본 FP16 압축)
ov_model = ov.convert_model(SAVED_MODEL_DIR, input=[[1, 224, 224, 3]])
ov.save_model(ov_model, IR_PATH)

for p in sorted(IR_PATH.parent.glob("leather_model.*")):
    print(f"{p.name:24s} {p.stat().st_size / 1024 / 1024:.2f} MB")

INFO:tensorflow:Assets written to: model\leather_saved_model\assets


INFO:tensorflow:Assets written to: model\leather_saved_model\assets


Saved artifact at 'model\leather_saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2122883324560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2122883325904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2121725693776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2121725692816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2121725692432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2121725693584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2121725694736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2121725693392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2121725695504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2121725693008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2121725696464: TensorSpe

## 셀 B — 변환 검증 (Keras vs OpenVINO)

In [5]:
img = Image.open("test_images/poke/000.png").convert("RGB").resize((224, 224))
arr = np.array(img, dtype=np.float32)[..., ::-1] - [103.939, 116.779, 123.68]  # VGG16 전처리
x = arr[np.newaxis, ...]

ov_out    = ov.Core().compile_model(IR_PATH, "CPU")(x)[0]
keras_out = model.predict(x, verbose=0)

print(f"Keras    : {float(keras_out.ravel()[0]):.4f}")
print(f"OpenVINO : {float(ov_out.ravel()[0]):.4f}")
print(f"차이      : {abs(float(keras_out.ravel()[0]) - float(ov_out.ravel()[0])):.5f}  (0.001 이하면 정상)")

Keras    : 0.9502
OpenVINO : 0.9501
차이      : 0.00017  (0.001 이하면 정상)
